# NB_CursorUsage_Bronze_To_Silver
**Source:** `AI_Medallion.Bronze.Cursor_CursorUsage`  
**Targets (reads):** `OktaUserforAI`, `HC_Silver_Historical`  
**Target:** `AI_Medallion.Silver.CursorUsage`  
**Run after:** NB_OktaUserforAI_Bronze_To_Silver, NB_HCHistorical_Bronze_To_Silver

In [1]:
import os
os.environ.setdefault("MEDALLION_DEBUG_READ", "1")
os.environ.setdefault("MEDALLION_LOCAL_TABLE_CACHE", "1")
os.environ.setdefault("MEDALLION_SCAN_CHUNK_ROWS", "75000")
os.environ.setdefault("MEDALLION_SCAN_RETRIES", "6")
# After kernel restart: run this cell, bootstrap, then read cells.
# read_table writes cache to ~/.fabric/medallion_cache/ (skip OneLake next time).
# Force OneLake: os.environ["MEDALLION_CACHE_REFRESH"] = "1"


'6'

In [2]:
%run ./common/bootstrap


Bootstrap OK: runtime=local mode=created spark=3.5.4 pyspark=3.5.4 path_reads=True onelake_read=deltalake-scan
spark.jars.packages=io.delta:delta-spark_2.12:3.2.0,org.apache.hadoop:hadoop-azure:3.3.4,com.azure:azure-storage-blob:12.25.1
OneLake reads: deltalake scan() + az login -> Spark (column mapping; avoids local PySpark abfss hang).
TABLE_CURSOR_BRONZE -> abfss://4cecee03-0629-4081-8ae8-b749ee04d43f@onelake.dfs.fabric.microsoft.com/1ee46463-2495-44f5-9dc9-d635d0067483/Tables/Bronze/cursor_usage
Tip: set MEDALLION_DEBUG_READ=1 before bootstrap to log read progress.
Tip: large tables — read_table() one at a time; transient Azure errors auto-retry.
Local table cache ON (default): parquet under ~/.fabric/medallion_cache/. Set MEDALLION_CACHE_REFRESH=1 to force OneLake. reload_io_helpers(globals()) after git pull — no kernel restart.
OneLake writes: deltalake + az login (merge/overwrite). Set MEDALLION_ONELAKE_WRITE=pyspark to use JVM abfss (often hangs).


In [3]:
bronze_df = read_table(TABLE_CURSOR_BRONZE)


read_table: cache hit AI_Medallion.Bronze.cursor_usage


In [4]:
okta = read_table(TABLE_OKTA_SILVER)


read_table: cache hit AI_Medallion.Silver.okta_user_for_ai


In [5]:
# Must finish without error before transform (HCUserNetwork cache schema fix on main).
hc = read_table(TABLE_HC_SILVER)


read_table: cache hit AI_Medallion.Silver.hc_silver_historical


In [6]:
bronze = bronze_df
user_norm = F.lower(F.trim(F.col("User")))


In [ ]:
apply_local_merge_spark_conf(spark)

base = (
    bronze.filter(is_valid_email(F.col("User")))
    .select(
        F.col("Date").cast("date").alias("CursorUsageDate"),
        user_norm.alias("CursorUsageUser"),
        F.col("`Service Account Name`").alias("CursorUsageServiceAccountName"),
        F.col("`Service Account ID`").alias("CursorUsageServiceAccountID"),
        F.col("`Cloud Agent ID`").alias("CursorUsageCloudAgentID"),
        F.col("`Automation ID`").alias("CursorUsageAutomationID"),
        F.col("Kind").alias("CursorUsageKind"),
        F.col("Model").alias("CursorUsageModel"),
        F.col("`Max Mode`").alias("CursorUsageMaxMode"),
        F.col("`Input (w/ Cache Write)`").cast("long").alias("CursorUsageInputWithCacheWrite"),
        F.col("`Input (w/o Cache Write)`").cast("long").alias("CursorUsageInputWithoutCacheWrite"),
        F.col("`Cache Read`").cast("long").alias("CursorUsageCacheRead"),
        F.col("`Output Tokens`").cast("long").alias("CursorUsageOutputTokens"),
        F.col("`Total Tokens`").cast("long").alias("CursorUsageTotalTokens"),
        F.col("Cost").cast("decimal(18,8)").alias("CursorUsageCost"),
        F.col("FileName").alias("CursorUsageFileName"),
        F.col("ExecutionDate").cast("timestamp").alias("CursorUsageIngestionDate"),
    )
    .withColumn(
        "CursorUsageMonthNo",
        (F.year("CursorUsageDate") * F.lit(100) + F.month("CursorUsageDate")).cast("int"),
    )
)

base = break_lineage_local(base, "cursor_base")

okta_slim = break_lineage_local(
    okta.select(
        "OktaUserId",
        "OktaUserEmail",
        "OktaUserFullEmail",
        "OktaUserEmployeeNumber",
        "OktaUserFullName",
        "OktaUserRegion",
        "OktaUserCountry",
    ),
    "okta_slim",
)

# Equi-joins only (OR joins -> BroadcastNestedLoopJoin + worker crash on staging write).
_okta_login = base.join(
    okta_slim, base.CursorUsageUser == okta_slim.OktaUserEmail, "inner"
)
_okta_full = base.join(
    okta_slim, base.CursorUsageUser == okta_slim.OktaUserFullEmail, "inner"
)
joined = _okta_login.unionByName(_okta_full).dropDuplicates()

joined = joined.withColumn(
    "CursorUsageKeyEmail",
    F.when(
        F.col("OktaUserEmail").isNotNull(),
        F.concat(
            F.col("CursorUsageMonthNo").cast("string"),
            F.lit("/"),
            F.col("OktaUserEmail"),
        ),
    ),
).withColumn(
    "CursorUsageKeyFullEmail",
    F.when(
        F.col("OktaUserFullEmail").isNotNull(),
        F.concat(
            F.col("CursorUsageMonthNo").cast("string"),
            F.lit("/"),
            F.col("OktaUserFullEmail"),
        ),
    ),
)

hc_slim = break_lineage_local(
    hc.select(
        F.col("HCKeyEmail"),
        F.col("HCBusinessUnit"),
        F.col("HCBusinessUnitCode"),
        F.col("HCDivision"),
        F.col("HCJobTitle"),
        F.col("HCOffice"),
        F.col("HCTalentSegment"),
        F.col("Month_No").alias("HCMonthNo"),
        F.col("OktaUserCountry").alias("HCOktaUserCountry"),
        F.col("HCDivisionINC"),
    ),
    "hc_slim",
)

_hc_attrs = [c for c in hc_slim.columns if c != "HCKeyEmail"]
_with_hc_email = joined.join(
    hc_slim, joined.CursorUsageKeyEmail == hc_slim.HCKeyEmail, "left"
)
_hc_alt = hc_slim.select(
    F.col("HCKeyEmail").alias("_hc_alt_key"),
    *[F.col(c).alias(f"_hc_alt_{c}") for c in _hc_attrs],
)
joined = _with_hc_email.join(
    _hc_alt, _with_hc_email.CursorUsageKeyFullEmail == _hc_alt._hc_alt_key, "left"
)
for _hc_col in _hc_attrs:
    joined = joined.withColumn(
        _hc_col, F.coalesce(F.col(_hc_col), F.col(f"_hc_alt_{_hc_col}"))
    )
joined = joined.drop(
    "_hc_alt_key", *[f"_hc_alt_{c}" for c in _hc_attrs]
).withColumn(
    "HCDivisionINC",
    F.coalesce(F.col("HCDivisionINC"), F.lit(HC_DIVISION_INC_DEFAULT)),
)

write_df= joined.distinct()

write_df=write_df.withColumn(
    fact_cursor_active_MERGE_KEY,
    cursor_usage_record_key()
    )

publish_merge_staging(write_df, TABLE_CURSOR_SILVER)


break_lineage_local: cursor_base -> C:\Users\abaitali\.fabric\merge_staging\_break\cursor_base
break_lineage_local: okta_slim -> C:\Users\abaitali\.fabric\merge_staging\_break\okta_slim
break_lineage_local: hc_slim -> C:\Users\abaitali\.fabric\merge_staging\_break\hc_slim
publish_merge_staging: AI_Medallion.Silver.cursor_usage -> C:\Users\abaitali\.fabric\merge_staging\AI_Medallion__Silver__cursor_usage


WindowsPath('C:/Users/abaitali/.fabric/merge_staging/AI_Medallion__Silver__cursor_usage')

In [ ]:
merge_incremental(
    spark,
    write_df,
    TABLE_CURSOR_SILVER,
    fact_cursor_active_MERGE_KEY,
)

merge_incremental: deltalake path abfss://4cecee03-0629-4081-8ae8-b749ee04d43f@onelake.dfs.fabric.microsoft.com/1ee46463-2495-44f5-9dc9-d635d0067483/Tables/Silver/cursor_usage
merge_incremental: chunk 1 (22720 rows)...
merge_incremental: deltalake merge done
